# Preparation for Gephi
To visualize the multiplex network, the study used the Gephi plugin "3D Splitter". This tool requires the data to be in a specific format. Based on the edge lists for the two layers, the following steps were performed:

All nodes across both layers were renamed in the format {state_name}_layer{number} to create a unified node list.
A new edge list was created, including both intra-layer edges and inter-layer edges (when states co-occur across the two layers).

*NOTE*
The attribute [Z] is required by the plugin to distinguish nodes from different layers.


In [1]:
import pandas as pd

In [6]:
df1 = pd.read_csv("../data/layer1/layer1_backbone_pos.csv")
df2 = pd.read_csv("../data/layer2/layer2_backbone_filtered.csv")

nodes_l1 = set(df1['s1']) | set(df1['s2'])
nodes_l2 = set(df2['source']) | set(df2['target'])
all_unique_nodes = nodes_l1 | nodes_l2

nodes_list = []

for name in nodes_l1:
    nodes_list.append({
        'Id': f"{name}_L1", 
        'Label': str(name), 
        'Layer': 'Layer 1', 
        'Level [Z]': 1,
    })

for name in nodes_l2:
    nodes_list.append({
        'Id': f"{name}_L0", 
        'Label': str(name), 
        'Layer': 'Layer 0', 
        'Level [Z]': 0
    })

nodes_df = pd.DataFrame(nodes_list)

edges_list = []

for _, row in df1.iterrows():
    edges_list.append({'Source': f"{row['s1']}_L1", 'Target': f"{row['s2']}_L1", 'Weight': row.get('avg_score', 1), 'Type': 'Intra-Layer'})

for _, row in df2.iterrows():
    edges_list.append({'Source': f"{row['source']}_L0", 'Target': f"{row['target']}_L0", 'Weight': row.get('weight', 1), 'Type': 'Intra-Layer'})

common_nodes = nodes_l1 & nodes_l2
for name in common_nodes:
    edges_list.append({
        'Source': f"{name}_L1", 
        'Target': f"{name}_L0", 
        'Weight': 1, 
        'Type': 'Inter-Layer'
    })

edges_df = pd.DataFrame(edges_list)

nodes_df.to_csv('../data/gephi/gephi_nodes_3d.csv', index=False)
edges_df.to_csv('../data/gephi/gephi_edges_3d.csv', index=False)